# Views: document matching & fact review

Source/doc-type mapping and the fact-comparison review layer. Split out of
`notebook_01_document_matching` (Cell 7) and `notebook_03_fact_comparison` (Cells 9 and 11).

## Why views live in their own notebooks
`CREATE OR REPLACE VIEW` drops and recreates the view, which makes the **executing identity its
owner**. A view owned by someone else that reads it (e.g. `gold_person_evidence_summary`) then
fails with `INSUFFICIENT_PERMISSIONS`. So view DDL is kept out of the data-refresh notebooks:
views are *definitions*, not data, and only need (re)creating when a definition changes.

**Run this notebook only when a view definition changes**, not as part of a routine refresh.
Run it as the account that owns the views (or follow it with
`ALTER VIEW <name> OWNER TO <owner>` for each view it recreates).

## Run order
1. `v_source_doctype_map` (a view over `ref_source_doctype_map`, so run `notebook_ref_source_doctype_map` first; must exist before `gold_source_coverage` is populated).
2. `gold_fact_comparison_reviewed` -- prerequisite: `silver_fact_conflict_review` (`notebook_fact_conflict_review_init`) and the `gold_fact_comparison` table.
3. `gold_housekeeping_tree_gap_occupations` -- reads (2).


## Cell 7 — Source/doc_type mapping view

Thin view over `genealogy.ref_source_doctype_map` (hand-maintained, never rebuilt; created and seeded by `notebook_ref_source_doctype_map`). Add or change titles in the table with `INSERT` / `UPDATE`, not here. The view only needs recreating if this definition changes.

In [0]:
%sql
CREATE OR REPLACE VIEW genealogy.v_source_doctype_map AS
SELECT source_title, doc_type_category
FROM genealogy.ref_source_doctype_map;

## Cell 9 — gold_fact_comparison_reviewed (applies review marks)

Prerequisite: `silver_fact_conflict_review` exists (notebook_fact_conflict_review_init).

Key: file_id + person_gedcom_id + fact_type + transcript_value (null-safe). The latest review
per key wins. `review_applies` is TRUE only while the tree value is unchanged since the
review (`reviewed_tree_value`); a changed tree value reopens the conflict, and a changed
transcript value or new file_id is simply a new key (open). Reviewed rows stay visible here.
`is_open_conflict` is the single definition of "conflict still needing attention".

`TREE_GAP` reviews (legitimate occupation missing from the tree) are the exception to the
tree-value snapshot rule: adding one occupation changes `tree_value`, which must not reopen the
person's other TREE_GAP rows as conflicts. They stay applied until the comparison is a MATCH.


In [0]:
%sql
CREATE OR REPLACE VIEW genealogy.gold_fact_comparison_reviewed AS
WITH latest_review AS (
  SELECT * EXCEPT (rn)
  FROM (
    SELECT r.*,
           ROW_NUMBER() OVER (
             PARTITION BY file_id, person_gedcom_id, fact_type, transcript_value
             ORDER BY reviewed_at DESC NULLS LAST
           ) AS rn
    FROM genealogy.silver_fact_conflict_review r
  )
  WHERE rn = 1
)
SELECT
  fc.*,
  rv.review_status,
  rv.reason_code        AS review_reason_code,
  rv.notes              AS review_notes,
  rv.reviewed_at,
  rv.reviewed_tree_value,
  COALESCE(rv.review_status IS NOT NULL
           AND (rv.reason_code = 'TREE_GAP' OR rv.reviewed_tree_value <=> fc.tree_value), FALSE)
                                                                              AS review_applies,
  (fc.status = 'CONFLICT'
   AND NOT COALESCE(rv.review_status IS NOT NULL
                    AND (rv.reason_code = 'TREE_GAP' OR rv.reviewed_tree_value <=> fc.tree_value), FALSE))
                                                                              AS is_open_conflict,
  -- Legitimate occupation the tree lacks: reviewed TREE_GAP and the comparison still isn't a MATCH.
  -- Clears on its own once the tree gains the occupation (status flips to MATCH on the next rebuild).
  COALESCE(rv.reason_code = 'TREE_GAP'
           AND fc.fact_type = 'occupation'
           AND fc.status IN ('CONFLICT', 'TRANSCRIPT_ONLY'), FALSE)           AS is_tree_gap
FROM genealogy.gold_fact_comparison fc
LEFT JOIN latest_review rv
  ON  rv.file_id          = fc.file_id
  AND rv.person_gedcom_id = fc.person_gedcom_id
  AND rv.fact_type        = fc.fact_type
  AND rv.transcript_value <=> fc.transcript_value;


## Cell 11 — gold_housekeeping_tree_gap_occupations (Housekeeping worklist)

One row per document occupation Ed has marked `TREE_GAP` that the tree still lacks, so he can add
a dated occupation in Ancestry. Rows vanish by themselves once the tree matches (no review step).
`NOT_AN_OCCUPATION` rows (students, club roles) are deliberately absent. Feeds
`SIGNAL_TRANSCRIPT_ONLY_FACTS` via `is_tree_gap`.


In [0]:
%sql
CREATE OR REPLACE VIEW genealogy.gold_housekeeping_tree_gap_occupations AS
SELECT
  fc.person_gedcom_id,
  fc.display_name,
  fc.file_id,
  fc.file_name,
  ot.year               AS document_year,
  fc.source_doc_type,
  fc.transcript_value   AS occupation_as_written,
  fc.tree_value         AS tree_occupations,
  fc.review_notes,
  fc.reviewed_at
FROM genealogy.gold_fact_comparison_reviewed fc
LEFT JOIN genealogy.ocr_transcriptions ot ON ot.file_id = fc.file_id
WHERE fc.is_tree_gap;


In [0]:
%sql
-- Open fact conflicts on direct ancestors only (ancestral_proximity = 0 in gold_ancestral_proximity,
-- per notebook 08: 0 = direct ancestor). Reads gold_fact_comparison_reviewed, so review marks apply.
-- Uses an uncorrelated IN subquery (correlated EXISTS over several tables trips a serverless optimizer bug).
CREATE OR REPLACE VIEW genealogy.gold_housekeeping_open_fact_conflicts_direct_ancestors AS
SELECT
  fc.person_gedcom_id,
  fc.display_name,
  fc.fact_type,
  fc.tree_value,
  fc.transcript_value,
  fc.file_id,
  fc.file_name,
  fc.source_doc_type,
  fc.conflict_severity,
  fc.review_status,
  fc.review_reason_code,
  fc.review_notes,
  fc.refreshed_at
FROM genealogy.gold_fact_comparison_reviewed fc
WHERE fc.is_open_conflict
  AND fc.person_gedcom_id IN (
    SELECT DISTINCT ap.person_id
    FROM genealogy.gold_ancestral_proximity ap
    WHERE ap.ancestral_proximity = 0
  );

In [0]:
%sql
-- Primary matches with no mention link (page_index / person_index NULL), for hand review.
-- Candidates are mentions in the same file whose name_raw contains the tree person's surname and first forename.
-- Nothing is linked automatically: review the image, then set override_person_gedcom_id (see the link-document-mentions skill).
-- Uses joins and uncorrelated subqueries only (correlated EXISTS across these tables trips a serverless optimizer bug).
CREATE OR REPLACE VIEW genealogy.gold_housekeeping_unlinked_primary_mentions AS
WITH prim AS (
  SELECT DISTINCT sdp.file_id, sdp.file_name, sdp.person_gedcom_id, sdp.display_name,
         sdp.match_method, sdp.match_confidence, p.surname, p.given_name, p.birth_year
  FROM genealogy.silver_document_person sdp
  JOIN genealogy.gold_person_life p ON p.person_gedcom_id = sdp.person_gedcom_id
  WHERE sdp.page_index IS NULL
    AND sdp.person_index IS NULL
    AND sdp.match_method IN ('EXACT', 'EXACT_MULTI', 'EXACT_ALIAS', 'EXACT_MULTI_ALIAS', 'FILENAME_ID')
),
cand AS (
  SELECT prim.file_id, prim.person_gedcom_id, m.person_index, m.name_raw, m.role_in_record,
         m.age_years, m.dob_raw, m.override_person_gedcom_id
  FROM prim
  JOIN genealogy.silver_transcript_person_mention m
    ON m.file_id = prim.file_id
   AND UPPER(m.name_raw) LIKE CONCAT('%', UPPER(prim.surname), '%')
   AND UPPER(m.name_raw) LIKE CONCAT('%', UPPER(SPLIT(prim.given_name, ' ')[0]), '%')
),
anc AS (
  SELECT DISTINCT person_id FROM genealogy.gold_ancestral_proximity WHERE ancestral_proximity = 0
)
SELECT
  prim.file_name,
  prim.file_id,
  prim.person_gedcom_id,
  prim.display_name,
  prim.birth_year,
  prim.match_method,
  prim.match_confidence,
  (anc.person_id IS NOT NULL) AS direct_ancestor,
  COUNT(cand.person_index) OVER (PARTITION BY prim.file_id, prim.person_gedcom_id) AS candidate_count,
  SUM(CASE WHEN cand.role_in_record IN ('Deceased', 'Husband', 'Wife', 'Head') THEN 1 ELSE 0 END)
    OVER (PARTITION BY prim.file_id, prim.person_gedcom_id) AS subject_role_candidates,
  cand.person_index AS candidate_person_index,
  cand.name_raw AS candidate_name_raw,
  cand.role_in_record AS candidate_role,
  cand.age_years AS candidate_age,
  cand.dob_raw AS candidate_dob,
  cand.override_person_gedcom_id AS candidate_override
FROM prim
LEFT JOIN cand ON cand.file_id = prim.file_id AND cand.person_gedcom_id = prim.person_gedcom_id
LEFT JOIN anc ON anc.person_id = prim.person_gedcom_id;